In [1]:
import os
import pandas as pd
import numpy as np
import warnings
import logging
import optuna
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Suppress noisy logs from Prophet/Stan/Optuna so our output stays clean
warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. LOAD AND PREPARE HOURLY DATA
# ==========================================
# Same data sources as the daily notebook, but here we keep the hourly granularity.
# The key difference: we filter to business hours (8am–4pm) BEFORE training
# so Prophet doesn't waste effort learning the overnight pattern (which is always zero).
print("Loading Hourly Data...")

data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

# --- SALES ---
# Filter to 08:00–16:00 (the store closes at 17:00, so the last ordering hour is 16:xx)
# This gives us 9 hourly buckets per day instead of 24
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
sales_df = sales_df[(sales_df['Date'].dt.hour >= 8) & (sales_df['Date'].dt.hour < 17)]
hourly_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()
product_cols = [c for c in hourly_sales.columns if c != 'Date']

# --- WEATHER (already hourly, so no aggregation needed) ---
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date'])
if 'weather_code' in weather_df.columns:
    weather_df['is_clear'] = (weather_df['weather_code'] == 0).astype(int)
    weather_df['is_cloudy'] = weather_df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    weather_df['is_rain'] = weather_df['weather_code'].isin(
        [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    weather_df['is_snow'] = weather_df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    weather_df = weather_df.drop(columns=['weather_code'])
hourly_weather = weather_df.copy()

# --- HOLIDAYS (daily data — will be broadcast to every hour of that day) ---
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date_Only'] = pd.to_datetime(holidays_df['Date']).dt.normalize()
daily_holidays = holidays_df.groupby('Date_Only').max(numeric_only=True).reset_index()

# --- EVENTS (same as holidays — daily flags broadcast to hourly) ---
events_df = pd.read_csv(events_path)
events_df['Date_Only'] = pd.to_datetime(events_df['Date']).dt.normalize()
daily_events = events_df.groupby('Date_Only').max(numeric_only=True).reset_index()

# --- BUILD REGRESSOR LIST ---
# Combine weather (hourly), holiday (daily), and event (daily) features
weather_features = [c for c in hourly_weather.columns if c not in ['Date', 'Time']]
holiday_features = [c for c in daily_holidays.columns if c != 'Date_Only']
event_features = [c for c in daily_events.columns if c != 'Date_Only']
regressors = list(dict.fromkeys(weather_features + holiday_features + event_features))

print(f"Products: {product_cols}")
print(f"Regressors ({len(regressors)}): {regressors}")
print(f"Hourly rows: {len(hourly_sales)}")
print(f"Date range: {hourly_sales['Date'].min()} to {hourly_sales['Date'].max()}")

# --- THE 74 PRODUCTS WE WANT TO FORECAST ---
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Loading Hourly Data...
Products: ['Almd & Aprct Bar', 'Apl & Blk Smthie', 'Avo & Hal Muffin', 'Avo, Egg & Bacon', 'Avo, Feta & Tom', 'Avocado on Toast', 'BD Curry Roll', 'BLT', 'Bacon', 'Bacon Bap', 'Bacon Egg Brioch', 'Bacon Waffle', 'Baked Beans', 'Baked Beans JP', 'Banana Bread', 'Bean Soldiers', 'Beef Burger', 'Beef Hors Foca', 'Beyond Burger', 'Big Breakfast', 'Black Pudding', 'Blk Forest Syrup', 'Bre Cran Toastie', 'Breakfast Bap', 'Breakfast Hash', 'Breakfast Muffin', 'Breakfast Wrap', 'Brie Bacon Crois', 'Brie Cranb Tstie', 'Brunch Burger', 'Buttd Mushrooms', 'Butter', 'Cheese & Bean JP', 'Cheese JP', 'Cheese Pizza', 'Cheese Sand PnM', 'Cheese Sandwich', 'Cheese Scone', 'Chick Chzo Tstie', 'Chick Flatbread', 'Chick Rice Bowl', 'Chicken Breast', 'Chicken Burger', 'Chicken Club', 'Chicken Goujons', 'Chicken Strips', 'Chicken Waffles', 'Chicken Wrap', 'Chickn Rice Bowl', 'Chilli Carne JP', 'Chips', 'Chorizo', 'Chorizo & Eggs', 'Chse Onion Tstie', 'Chse Tom Toastie', 'Cinnamon Dust

In [2]:
# ==========================================
# 2. DATA-DRIVEN PRODUCT CLUSTERING (HOURLY)
# ==========================================
#
# WHY CLUSTER?
# Same reason as the daily notebook: Bacon Bap's tuned params don't suit every
# product. But for the hourly model there's an extra benefit — we can also tune
# the daily_fourier order per cluster. A breakfast item (peaks 8-10am) needs a
# different intra-day shape than a lunch item (peaks 12-1pm). Clustering by
# morning_ratio lets Optuna find the right Fourier complexity for each group.
#
# FEATURES (6 total — same 5 as daily + 1 new hourly-specific one):
#   - mean_hourly_volume:  Average units sold per hour (high vs low seller)
#   - coeff_of_variation:  How much hourly sales vary (stable vs unpredictable)
#   - zero_hour_fraction:  What % of hours had zero sales (sparse vs consistent)
#   - weekend_ratio:       Weekend vs weekday demand ratio
#   - temp_correlation:    Correlation between sales and temperature
#   - morning_ratio (NEW): What % of sales happen 8-11am vs 12-4pm
#                          High = breakfast item, Low = lunch item
#                          This drives the daily_fourier tuning per cluster

print("Building Data-Driven Product Clusters (Hourly)...")

# Only use training data (before Oct 2025) for clustering
train_end = pd.to_datetime('2025-10-01')
train_sales = hourly_sales[hourly_sales['Date'] <= train_end].copy()

# Merge temperature for correlation calculation
train_merged = train_sales.merge(hourly_weather[['Date', 'apparent_temperature']], on='Date', how='left')
train_merged['dow'] = train_merged['Date'].dt.dayofweek   # Monday=0, Sunday=6
train_merged['hour'] = train_merged['Date'].dt.hour        # 8, 9, 10, ..., 16

product_features = []
for product in PRODUCTS_TO_FORECAST:
    if product not in train_sales.columns:
        continue
    series = train_sales[product].values.astype(float)
    
    # Average hourly sales
    mean_vol = np.mean(series)
    
    # Coefficient of variation (std / mean) — how erratic is demand?
    std_vol = np.std(series)
    cv = std_vol / mean_vol if mean_vol > 0 else 999.0
    
    # Fraction of hours with zero sales
    zero_frac = np.mean(series == 0)
    
    # Weekend vs weekday ratio
    weekend_mask = train_merged['dow'].isin([5, 6])
    wkend_mean = train_merged.loc[weekend_mask, product].mean()
    wkday_mean = train_merged.loc[~weekend_mask, product].mean()
    weekend_ratio = wkend_mean / wkday_mean if wkday_mean > 0 else 1.0
    
    # Temperature correlation — positive means more sales on warmer days
    temp_corr = train_merged[[product, 'apparent_temperature']].corr().iloc[0, 1]
    if np.isnan(temp_corr):
        temp_corr = 0.0
    
    # Morning ratio — what fraction of total sales happen in the morning (8-11am)?
    # Bacon Bap might be 80% morning, Chicken Club might be 20% morning
    # This tells us whether the product is breakfast-heavy or lunch-heavy
    morning_mask = train_merged['hour'].isin([8, 9, 10, 11])
    afternoon_mask = train_merged['hour'].isin([12, 13, 14, 15, 16])
    morning_sales = train_merged.loc[morning_mask, product].sum()
    afternoon_sales = train_merged.loc[afternoon_mask, product].sum()
    total = morning_sales + afternoon_sales
    morning_ratio = morning_sales / total if total > 0 else 0.5
    
    product_features.append({
        'product': product,
        'mean_hourly_volume': mean_vol,
        'coeff_of_variation': cv,
        'zero_hour_fraction': zero_frac,
        'weekend_ratio': weekend_ratio,
        'temp_correlation': temp_corr,
        'morning_ratio': morning_ratio
    })

features_df = pd.DataFrame(product_features)
print(f"Computed demand-shape features for {len(features_df)} products")

# ==========================================
# RUN KMEANS TO GROUP PRODUCTS INTO 4 CLUSTERS
# ==========================================
N_CLUSTERS = 4

# Standardise so KMeans treats all features equally
feature_cols = ['mean_hourly_volume', 'coeff_of_variation', 'zero_hour_fraction',
                'weekend_ratio', 'temp_correlation', 'morning_ratio']
X = features_df[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
features_df['cluster_id'] = kmeans.fit_predict(X_scaled)

# Pick the highest-volume product per cluster as the "hero" for tuning
PRODUCT_CLUSTER_MAP = {}   # product name -> cluster number
CLUSTER_HEROES = {}        # cluster number -> hero product name

for cid in range(N_CLUSTERS):
    cluster_products = features_df[features_df['cluster_id'] == cid]
    hero = cluster_products.sort_values('mean_hourly_volume', ascending=False).iloc[0]
    CLUSTER_HEROES[cid] = hero['product']
    for _, row in cluster_products.iterrows():
        PRODUCT_CLUSTER_MAP[row['product']] = cid

# Print cluster summary
print(f"\n{'='*60}")
print(f"  CLUSTER SUMMARY ({N_CLUSTERS} clusters)")
print(f"{'='*60}")
for cid in range(N_CLUSTERS):
    members = [p for p, c in PRODUCT_CLUSTER_MAP.items() if c == cid]
    hero = CLUSTER_HEROES[cid]
    hero_row = features_df[features_df['product'] == hero].iloc[0]
    print(f"\n  Cluster {cid} ({len(members)} products)")
    print(f"    Hero (used for tuning): {hero}")
    print(f"    Hero stats: {hero_row['mean_hourly_volume']:.2f} avg/hr, "
          f"CV={hero_row['coeff_of_variation']:.2f}, "
          f"Morning%={hero_row['morning_ratio']:.0%}, "
          f"Temp corr={hero_row['temp_correlation']:.2f}")
    print(f"    All members: {', '.join(members[:10])}{'...' if len(members) > 10 else ''}")


Building Data-Driven Product Clusters (Hourly)...
Computed demand-shape features for 64 products

  CLUSTER SUMMARY (4 clusters)

  Cluster 0 (42 products)
    Hero (used for tuning): Big Breakfast
    Hero stats: 0.93 avg/hr, CV=1.84, Morning%=62%, Temp corr=-0.03
    All members: Avo & Hal Muffin, Avo, Egg & Bacon, Avo, Feta & Tom, Avocado on Toast, Bacon, Bacon Egg Brioch, Bacon Waffle, Baked Beans, Bean Soldiers, Big Breakfast...

  Cluster 1 (6 products)
    Hero (used for tuning): Bakery
    Hero stats: 5.84 avg/hr, CV=0.75, Morning%=33%, Temp corr=0.03
    All members: Bacon Bap, Sausage Bap, The Breakfast, Toasted Teacake, Bakery, White Toast Bread

  Cluster 2 (2 products)
    Hero (used for tuning): Festive Stack
    Hero stats: 0.00 avg/hr, CV=999.00, Morning%=50%, Temp corr=0.00
    All members: Festive Stack, Tuna Mayo Mix

  Cluster 3 (14 products)
    Hero (used for tuning): Tuna Panini
    Hero stats: 0.22 avg/hr, CV=2.67, Morning%=15%, Temp corr=-0.03
    All members: 

In [3]:
# ==========================================
# 3. CLUSTER-BASED OPTUNA TUNING (HOURLY)
# ==========================================
#
# Same cluster-based approach as the daily notebook, with one extra tunable parameter:
#
#   daily_fourier — controls the complexity of the intra-day (within-day) pattern.
#     - Low Fourier (3-4): simple curve, one peak per day (good for breakfast-only items)
#     - High Fourier (6-8): complex curve, can capture multiple peaks (good for items
#       sold across both breakfast and lunch with a dip in between)
#
# By tuning daily_fourier per cluster, a breakfast-heavy cluster can learn
# "sales peak at 9am then tail off" while a lunch-heavy cluster can learn
# "sales are low at 8am, peak at 12pm, then tail off."
#
# FACTORY PATTERN: Same closure-bug fix as the daily notebook — make_objective()
# creates a fresh copy of training data for each cluster.

print("Running Cluster-Based Optuna Tuning for HOURLY Prophet...")
print(f"Tuning {N_CLUSTERS} clusters × 20 trials each = {N_CLUSTERS * 20} total Optuna trials")

cluster_best_params = {}

def make_objective(train_data, regressor_list):
    """Create an Optuna objective function with its own copy of the training data.
    
    Without this factory pattern, Python's closure would cause all clusters to
    share the same training data (a subtle but critical bug).
    """
    _train = train_data.copy()
    _regs = list(regressor_list)
    
    def objective(trial):
        # Same hyperparameters as daily, plus daily_fourier for the intra-day shape
        cps = trial.suggest_float('changepoint_prior_scale', 0.001, 0.5, log=True)
        sps = trial.suggest_float('seasonality_prior_scale', 0.01, 10.0, log=True)
        hps = trial.suggest_float('holidays_prior_scale', 0.01, 10.0, log=True)
        cpr = trial.suggest_float('changepoint_range', 0.7, 0.95)
        n_cp = trial.suggest_int('n_changepoints', 15, 50)
        s_mode = trial.suggest_categorical('seasonality_mode', ['additive', 'multiplicative'])
        daily_fourier = trial.suggest_int('daily_fourier', 3, 8)

        m = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,  # We add our own custom daily seasonality below
            changepoint_prior_scale=cps,
            seasonality_prior_scale=sps,
            holidays_prior_scale=hps,
            changepoint_range=cpr,
            n_changepoints=n_cp,
            seasonality_mode=s_mode
        )
        # Custom daily seasonality with tunable Fourier order
        # period=1 means "1 day", fourier_order controls how many sine/cosine terms
        m.add_seasonality(name='daily_business', period=1, fourier_order=daily_fourier)
        m.add_country_holidays(country_name='UK')
        for reg in _regs:
            m.add_regressor(reg, standardize=True)

        m.fit(_train)

        try:
            df_cv = cross_validation(m, initial='730 days', period='30 days', horizon='30 days', parallel='threads')
            df_p = performance_metrics(df_cv, rolling_window=1)
            return df_p['rmse'].values[0]
        except Exception:
            return float('inf')
    
    return objective


# --- Run Optuna for each cluster's hero product ---
for cluster_id, hero_product in CLUSTER_HEROES.items():
    print(f"\n>> Tuning Cluster {cluster_id} (Hero: {hero_product})")
    
    # Build training dataframe for this hero
    # Note: holidays and events are daily, so we merge on Date_Only then drop it
    tune_df = hourly_sales[['Date', hero_product]].copy()
    tune_df['Date_Only'] = tune_df['Date'].dt.normalize()
    tune_df = tune_df.merge(hourly_weather, on='Date', how='left')
    tune_df = tune_df.merge(daily_holidays, on='Date_Only', how='left')
    tune_df = tune_df.merge(daily_events[['Date_Only'] + event_features], on='Date_Only', how='left')
    tune_df = tune_df.drop(columns=['Date_Only'])
    
    # Fill NaNs and clip binary flags
    num_cols = tune_df.select_dtypes(include=[np.number]).columns
    tune_df[num_cols] = tune_df[num_cols].fillna(0)
    flag_cols = [c for c in regressors if c.startswith('is_')]
    if flag_cols:
        tune_df[flag_cols] = tune_df[flag_cols].clip(upper=1)
    
    # Prophet expects 'ds' and 'y'
    tune_df = tune_df.rename(columns={'Date': 'ds', hero_product: 'y'})
    tune_train = tune_df[tune_df['ds'] <= train_end].copy()

    # Run 20 Optuna trials for this cluster
    objective_fn = make_objective(tune_train, regressors)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_fn, n_trials=20)
    
    cluster_best_params[cluster_id] = study.best_params
    bp = study.best_params
    print(f"  Best CV RMSE: {study.best_value:.4f}")
    print(f"  Seasonality: {bp['seasonality_mode']}, "
          f"Daily Fourier: {bp['daily_fourier']}, "
          f"CPS: {bp['changepoint_prior_scale']:.4f}")

# --- Summary ---
print(f"\n{'='*60}")
print("  TUNING COMPLETE — What each cluster got:")
print(f"{'='*60}")
for cid, params in cluster_best_params.items():
    hero = CLUSTER_HEROES[cid]
    print(f"  Cluster {cid} (tuned on {hero}):")
    print(f"    seasonality_mode={params['seasonality_mode']}, "
          f"daily_fourier={params['daily_fourier']}, "
          f"changepoint_prior_scale={params['changepoint_prior_scale']:.4f}")


Running Cluster-Based Optuna Tuning for HOURLY Prophet...
Tuning 4 clusters × 20 trials each = 80 total Optuna trials

>> Tuning Cluster 0 (Hero: Big Breakfast)


14:25:37 - cmdstanpy - INFO - Chain [1] start processing
14:25:40 - cmdstanpy - INFO - Chain [1] done processing
14:25:56 - cmdstanpy - INFO - Chain [1] start processing
14:25:57 - cmdstanpy - INFO - Chain [1] done processing
14:25:57 - cmdstanpy - INFO - Chain [1] start processing
14:25:58 - cmdstanpy - INFO - Chain [1] start processing
14:25:58 - cmdstanpy - INFO - Chain [1] start processing
14:25:58 - cmdstanpy - INFO - Chain [1] start processing
14:25:58 - cmdstanpy - INFO - Chain [1] start processing
14:25:58 - cmdstanpy - INFO - Chain [1] done processing
14:25:59 - cmdstanpy - INFO - Chain [1] done processing
14:26:00 - cmdstanpy - INFO - Chain [1] start processing
14:26:00 - cmdstanpy - INFO - Chain [1] done processing
14:26:00 - cmdstanpy - INFO - Chain [1] done processing
14:26:00 - cmdstanpy - INFO - Chain [1] start processing
14:26:00 - cmdstanpy - INFO - Chain [1] start processing
14:26:00 - cmdstanpy - INFO - Chain [1] done processing
14:26:00 - cmdstanpy - INFO - Chain [1

  Best CV RMSE: 1.7204
  Seasonality: multiplicative, Daily Fourier: 4, CPS: 0.0168

>> Tuning Cluster 1 (Hero: Bakery)


14:40:57 - cmdstanpy - INFO - Chain [1] start processing
14:40:58 - cmdstanpy - INFO - Chain [1] done processing
14:41:13 - cmdstanpy - INFO - Chain [1] start processing
14:41:13 - cmdstanpy - INFO - Chain [1] done processing
14:41:13 - cmdstanpy - INFO - Chain [1] start processing
14:41:13 - cmdstanpy - INFO - Chain [1] start processing
14:41:14 - cmdstanpy - INFO - Chain [1] start processing
14:41:14 - cmdstanpy - INFO - Chain [1] done processing
14:41:14 - cmdstanpy - INFO - Chain [1] done processing
14:41:15 - cmdstanpy - INFO - Chain [1] start processing
14:41:15 - cmdstanpy - INFO - Chain [1] done processing
14:41:15 - cmdstanpy - INFO - Chain [1] start processing
14:41:15 - cmdstanpy - INFO - Chain [1] start processing
14:41:16 - cmdstanpy - INFO - Chain [1] start processing
14:41:16 - cmdstanpy - INFO - Chain [1] start processing
14:41:16 - cmdstanpy - INFO - Chain [1] start processing
14:41:16 - cmdstanpy - INFO - Chain [1] start processing
14:41:16 - cmdstanpy - INFO - Chain 

  Best CV RMSE: 3.3362
  Seasonality: additive, Daily Fourier: 6, CPS: 0.0077

>> Tuning Cluster 2 (Hero: Festive Stack)
  Best CV RMSE: 0.0000
  Seasonality: multiplicative, Daily Fourier: 3, CPS: 0.0416

>> Tuning Cluster 3 (Hero: Tuna Panini)


14:55:12 - cmdstanpy - INFO - Chain [1] start processing
14:55:15 - cmdstanpy - INFO - Chain [1] done processing
14:55:32 - cmdstanpy - INFO - Chain [1] start processing
14:55:32 - cmdstanpy - INFO - Chain [1] start processing
14:55:32 - cmdstanpy - INFO - Chain [1] start processing
14:55:33 - cmdstanpy - INFO - Chain [1] start processing
14:55:34 - cmdstanpy - INFO - Chain [1] start processing
14:55:34 - cmdstanpy - INFO - Chain [1] start processing
14:55:35 - cmdstanpy - INFO - Chain [1] start processing
14:55:35 - cmdstanpy - INFO - Chain [1] start processing
14:55:35 - cmdstanpy - INFO - Chain [1] done processing
14:55:35 - cmdstanpy - INFO - Chain [1] start processing
14:55:35 - cmdstanpy - INFO - Chain [1] start processing
14:55:35 - cmdstanpy - INFO - Chain [1] start processing
14:55:35 - cmdstanpy - INFO - Chain [1] done processing
14:55:36 - cmdstanpy - INFO - Chain [1] start processing
14:55:36 - cmdstanpy - INFO - Chain [1] start processing
14:55:38 - cmdstanpy - INFO - Chai

  Best CV RMSE: 0.5535
  Seasonality: multiplicative, Daily Fourier: 8, CPS: 0.4640

  TUNING COMPLETE — What each cluster got:
  Cluster 0 (tuned on Big Breakfast):
    seasonality_mode=multiplicative, daily_fourier=4, changepoint_prior_scale=0.0168
  Cluster 1 (tuned on Bakery):
    seasonality_mode=additive, daily_fourier=6, changepoint_prior_scale=0.0077
  Cluster 2 (tuned on Festive Stack):
    seasonality_mode=multiplicative, daily_fourier=3, changepoint_prior_scale=0.0416
  Cluster 3 (tuned on Tuna Panini):
    seasonality_mode=multiplicative, daily_fourier=8, changepoint_prior_scale=0.4640


In [4]:
# ==========================================
# 4. TRAIN & FORECAST ALL 74 PRODUCTS (HOURLY)
# ==========================================
#
# Every product gets a full Prophet model with its cluster's tuned params —
# no products are skipped. This keeps the comparison fair with XGBoost and LSTM,
# which also forecast all 74 products.
#
# Each product gets:
#   - The seasonality_mode from its cluster (additive vs multiplicative)
#   - The daily_fourier from its cluster (simple vs complex intra-day shape)
#   - The changepoint/holiday priors from its cluster
#
# After forecasting hourly, we'll roll up to daily totals in the next cell
# so we can compare against the daily Prophet and other daily models.

print("\nTraining Hourly Prophet for ALL 74 products using cluster-specific params...")

test_start = pd.to_datetime('2025-11-01 00:00:00')
test_end = pd.to_datetime('2025-11-30 23:59:59')

all_hourly_predictions = []

for product in PRODUCTS_TO_FORECAST:
    # Look up this product's cluster and its tuned params
    cluster_id = PRODUCT_CLUSTER_MAP.get(product, 0)
    params = cluster_best_params[cluster_id]

    # --- Build hourly feature dataframe ---
    # Merge sales with hourly weather, then broadcast daily holidays/events
    pdf = hourly_sales[['Date', product]].copy()
    pdf['Date_Only'] = pdf['Date'].dt.normalize()
    pdf = pdf.merge(hourly_weather, on='Date', how='left')
    pdf = pdf.merge(daily_holidays, on='Date_Only', how='left')
    pdf = pdf.merge(daily_events[['Date_Only'] + event_features], on='Date_Only', how='left')
    pdf = pdf.drop(columns=['Date_Only'])

    # Clean up NaNs and binary flags
    num_cols = pdf.select_dtypes(include=[np.number]).columns
    pdf[num_cols] = pdf[num_cols].fillna(0)
    flag_cols = [c for c in regressors if c.startswith('is_')]
    if flag_cols:
        pdf[flag_cols] = pdf[flag_cols].clip(upper=1)

    prophet_df = pdf.rename(columns={'Date': 'ds', product: 'y'})

    # Split into training (up to Oct) and test (November)
    p_train = prophet_df[prophet_df['ds'] <= train_end].copy()
    p_test = prophet_df[(prophet_df['ds'] >= test_start) & (prophet_df['ds'] <= test_end)].copy()

    if len(p_test) == 0:
        continue

    # --- Build Prophet model with THIS cluster's tuned params ---
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        holidays_prior_scale=params['holidays_prior_scale'],
        changepoint_range=params['changepoint_range'],
        n_changepoints=params['n_changepoints'],
        seasonality_mode=params['seasonality_mode']
    )
    # Custom intra-day seasonality with this cluster's Fourier order
    m.add_seasonality(name='daily_business', period=1, fourier_order=params['daily_fourier'])
    m.add_country_holidays(country_name='UK')
    for reg in regressors:
        m.add_regressor(reg, standardize=True)

    m.fit(p_train)

    # Generate hourly predictions for November
    future = p_test[['ds'] + regressors].copy()
    forecast = m.predict(future)

    # Store results — clip negatives to 0
    result = p_test[['ds', 'y']].copy()
    result['Hourly_Prophet_Prediction'] = np.maximum(forecast['yhat'].values, 0)
    result['Product_Name'] = product
    result['Cluster'] = str(cluster_id)
    all_hourly_predictions.append(result)
    print(f"  {product}: Cluster {cluster_id} "
          f"({params['seasonality_mode']}, fourier={params['daily_fourier']}) done")

hourly_eval_df = pd.concat(all_hourly_predictions, ignore_index=True)
print(f"\nTotal hourly predictions: {len(hourly_eval_df)} rows")



Training Hourly Prophet for ALL 74 products using cluster-specific params...


15:11:39 - cmdstanpy - INFO - Chain [1] start processing
15:11:41 - cmdstanpy - INFO - Chain [1] done processing


  Avo & Hal Muffin: Cluster 0 (multiplicative, fourier=4) done


15:11:43 - cmdstanpy - INFO - Chain [1] start processing
15:11:44 - cmdstanpy - INFO - Chain [1] done processing


  Avo, Egg & Bacon: Cluster 0 (multiplicative, fourier=4) done


...

  Porridge: Cluster 0 (multiplicative, fourier=4) done


15:15:31 - cmdstanpy - INFO - Chain [1] start processing
15:15:33 - cmdstanpy - INFO - Chain [1] done processing


  Sourdough Toast Bread: Cluster 0 (multiplicative, fourier=4) done


15:15:34 - cmdstanpy - INFO - Chain [1] start processing
15:15:36 - cmdstanpy - INFO - Chain [1] done processing


  Multiseed Toast Bread: Cluster 0 (multiplicative, fourier=4) done

Total hourly predictions: 17280 rows


In [5]:
# ==========================================
# 5. ROLL UP HOURLY → DAILY + BUILD NAIVE BASELINE
# ==========================================
#
# Prophet predicted at the hourly level, but our evaluation metrics and model
# comparison are all at the daily level. So we sum up the 9 hourly predictions
# per day to get a single daily total per product.
#
# We also build a naive baseline (yesterday's daily total) so we can calculate
# MASE — which tells us whether the model is actually better than just using
# yesterday's number as today's forecast.

print("Rolling up hourly predictions to daily totals...")

hourly_eval_df['Date_Only'] = hourly_eval_df['ds'].dt.date

# Sum all hours of each day into one daily total per product
daily_rollup = hourly_eval_df.groupby(['Product_Name', 'Date_Only', 'Cluster']).agg({
    'y': 'sum',
    'Hourly_Prophet_Prediction': 'sum'
}).reset_index()
daily_rollup = daily_rollup.rename(columns={
    'y': 'Sales',
    'Hourly_Prophet_Prediction': 'Prophet_Prediction'
})

# Build naive baseline: for each product, yesterday's daily total
daily_rollup = daily_rollup.sort_values(['Product_Name', 'Date_Only']).reset_index(drop=True)
daily_rollup['sales_1_day_ago'] = daily_rollup.groupby('Product_Name')['Sales'].shift(1)

# For the very first test day, we don't have a "yesterday" in the test set
# so we grab the last known daily total from training data
for product in product_cols:
    p_mask = daily_rollup['Product_Name'] == product
    if daily_rollup.loc[p_mask, 'sales_1_day_ago'].isna().any():
        # Get the last day's total from training data
        prod_hourly = hourly_sales[hourly_sales['Date'] < test_start][['Date', product]].copy()
        prod_hourly['_date'] = prod_hourly['Date'].dt.date
        last_daily = prod_hourly.groupby('_date')[product].sum()
        if len(last_daily) > 0:
            daily_rollup.loc[p_mask & daily_rollup['sales_1_day_ago'].isna(), 'sales_1_day_ago'] = last_daily.iloc[-1]

daily_rollup['sales_1_day_ago'] = daily_rollup['sales_1_day_ago'].fillna(0)

print(f"Daily rollup: {len(daily_rollup)} rows")


Rolling up hourly predictions to daily totals...
Daily rollup: 1920 rows


In [6]:
# ==========================================
# 6. METRIC FUNCTION
# ==========================================
# Same metric function as the daily notebook and the XGBoost/LSTM notebooks.
# See daily notebook Cell 5 for detailed explanations of each metric.

def calculate_north_star_metrics(evaluation_slice):
    """Calculate WAPE, MASE, MAE, Bias and legacy metrics for a slice of predictions."""
    if len(evaluation_slice) == 0:
        return {m: 0 for m in ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE']}

    actual = evaluation_slice['Sales'].values.astype(float)
    predicted = evaluation_slice['Prophet_Prediction'].values.astype(float)
    naive = evaluation_slice['sales_1_day_ago'].values.astype(float)

    # WAPE: total absolute error / total actual sales
    total_abs_error = np.sum(np.abs(actual - predicted))
    total_actual = np.sum(np.abs(actual))
    wape = total_abs_error / total_actual if total_actual > 0 else np.nan

    # MASE: our MAE / naive baseline MAE — below 1.0 means we beat naive
    mae = mean_absolute_error(actual, predicted)
    naive_mae = mean_absolute_error(actual, naive)
    mase = mae / naive_mae if naive_mae > 0 else np.nan

    # Bias: average (predicted - actual) — positive = over-predicting
    bias = np.mean(predicted - actual)

    # Legacy metrics
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)

    nonzero = actual > 0
    mape = mean_absolute_percentage_error(actual[nonzero], predicted[nonzero]) if nonzero.sum() > 0 else np.nan

    abs_err = np.abs(predicted - actual)
    denom = (np.abs(actual) + np.abs(predicted)) / 2.0
    valid = denom > 0
    smape = np.mean(abs_err[valid] / denom[valid]) if valid.sum() > 0 else np.nan

    return {'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': bias,
            'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}


In [7]:
# ==========================================
# 7. PER-PRODUCT DETAILED SCORECARDS
# ==========================================
# Same 3-horizon scorecard as daily: 1-Day, 1-Week, 1-Month
# Plus a reality-check table showing actual vs predicted for the first 10 days

t1 = pd.to_datetime('2025-11-02').date()
t7 = pd.to_datetime('2025-11-08').date()
t30 = pd.to_datetime('2025-11-30').date()

all_products = sorted(daily_rollup['Product_Name'].unique())

for product in all_products:
    p_daily = daily_rollup[daily_rollup['Product_Name'] == product].copy()
    p_daily = p_daily.sort_values('Date_Only').reset_index(drop=True)

    df_1d = p_daily[p_daily['Date_Only'] == t1]
    df_1w = p_daily[(p_daily['Date_Only'] >= t1) & (p_daily['Date_Only'] <= t7)]
    df_1m = p_daily[(p_daily['Date_Only'] >= t1) & (p_daily['Date_Only'] <= t30)]

    comparison_df = pd.DataFrame({
        'Metric': ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE'],
        '1-Day (Nov 2)': list(calculate_north_star_metrics(df_1d).values()),
        '1-Week (Nov 2-8)': list(calculate_north_star_metrics(df_1w).values()),
        '1-Month (Nov 2-30)': list(calculate_north_star_metrics(df_1m).values())
    })

    for col in comparison_df.columns[1:]:
        comparison_df[col] = comparison_df[col].apply(
            lambda x: f"{float(x):.4f}" if not np.isnan(x) else "N/A"
        )

    cluster_label = PRODUCT_CLUSTER_MAP.get(product, '?')
    print(f"\n======================================================")
    print(f"  HOURLY PROPHET METRICS (ROLLED UP): {product} [Cluster: {cluster_label}]")
    print(f"======================================================")
    print(comparison_df.to_string(index=False))

    p_daily['Predicted_Rounded'] = p_daily['Prophet_Prediction'].round().astype(int)
    p_daily['Mistake_Amount'] = (p_daily['Sales'] - p_daily['Predicted_Rounded']).abs()

    print(f"\n  REALITY CHECK: {product}")
    print(f"  --------------------------------------------------")
    print(p_daily[['Date_Only', 'Sales', 'Predicted_Rounded', 'Mistake_Amount']].head(10).to_string(index=False))



  HOURLY PROPHET METRICS (ROLLED UP): Avo & Hal Muffin [Cluster: 0]
Metric 1-Day (Nov 2) 1-Week (Nov 2-8) 1-Month (Nov 2-30)
  WAPE           N/A              N/A                N/A
  MASE           N/A              N/A                N/A
   MAE        0.0000           0.0000             0.0001
  Bias        0.0000           0.0000             0.0001
   MSE        0.0000           0.0000             0.0000
  RMSE        0.0000           0.0000             0.0002
  MAPE           N/A              N/A                N/A
 SMAPE           N/A           2.0000             2.0000

  REALITY CHECK: Avo & Hal Muffin
  --------------------------------------------------
 Date_Only  Sales  Predicted_Rounded  Mistake_Amount
2025-11-01      0                  0               0
2025-11-02      0                  0               0
2025-11-03      0                  0               0
2025-11-04      0                  0               0
2025-11-05      0                  0               0
2025-11-06  

In [8]:
# ==========================================
# 8. LEADERBOARD + AGGREGATE METRICS
# ==========================================
# Three views: per-product leaderboard, overall aggregate, per-cluster breakdown.
# See daily notebook Cell 7 for detailed explanations of each section.

print("\n==================================================")
print("  PER-PRODUCT NORTH STAR METRICS")
print("  Sorted by Bias (most under-predicted first)")
print("==================================================")

target_date_1 = pd.to_datetime('2025-11-02').date()
target_date_30 = pd.to_datetime('2025-11-30').date()
per_product_metrics = []
for product_name in all_products:
    product_data = daily_rollup[daily_rollup['Product_Name'] == product_name].copy()
    product_month_data = product_data[(product_data['Date_Only'] >= target_date_1) & (product_data['Date_Only'] <= target_date_30)]
    product_metrics = calculate_north_star_metrics(product_month_data)
    product_metrics['Product'] = product_name
    product_metrics['Cluster'] = str(PRODUCT_CLUSTER_MAP.get(product_name, '?'))
    per_product_metrics.append(product_metrics)

product_leaderboard_df = pd.DataFrame(per_product_metrics)
product_leaderboard_df = product_leaderboard_df[['Product', 'Cluster', 'WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']]
product_leaderboard_df = product_leaderboard_df.sort_values('Bias')

for col in ['WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']:
    product_leaderboard_df[col] = product_leaderboard_df[col].apply(
        lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A"
    )

print(product_leaderboard_df.to_string(index=False))

print("\n  HOW TO READ THIS TABLE:")
print("  - WAPE < 0.20  → Good — forecast is within 20% of actual volume")
print("  - MASE < 1.00  → Model beats naive baseline (yesterday's sales)")
print("  - MASE > 1.00  → Model is WORSE than just using yesterday's number")
print("  - Bias < 0     → Under-predicting → stock-out risk")
print("  - Bias > 0     → Over-predicting → waste risk")

# --- OVERALL AGGREGATE ---
all_month = daily_rollup[(daily_rollup['Date_Only'] >= target_date_1) & (daily_rollup['Date_Only'] <= target_date_30)].copy()
overall = calculate_north_star_metrics(all_month)

print("\n==================================================")
print("  OVERALL: Prophet Hourly Clustered (All 74 Products)")
print("==================================================")
print(f"  WAPE:  {overall['WAPE']:.4f}  ← % of total inventory wrong")
print(f"  MASE:  {overall['MASE']:.4f}  ← {'BEATING' if overall['MASE'] < 1 else 'LOSING TO'} naive baseline")
print(f"  MAE:   {overall['MAE']:.4f}  ← avg units off per product per day")
print(f"  Bias:  {overall['Bias']:.4f}  ← {'under-predicting (stock-out risk)' if overall['Bias'] < 0 else 'over-predicting (waste risk)'}")
print("  ---")
for m_name in ['RMSE', 'MAPE', 'SMAPE']:
    val = overall[m_name]
    print(f"  {m_name:>6s}: {val:.4f}" if not np.isnan(val) else f"  {m_name:>6s}: N/A")

# --- PER-CLUSTER BREAKDOWN ---
print("\n==================================================")
print("  PER-CLUSTER BREAKDOWN")
print("==================================================")
for cluster_label in sorted(daily_rollup['Cluster'].unique()):
    cluster_month = all_month[all_month['Cluster'] == cluster_label]
    if len(cluster_month) == 0:
        continue
    cm = calculate_north_star_metrics(cluster_month)
    n_products = cluster_month['Product_Name'].nunique()
    hero = CLUSTER_HEROES.get(int(cluster_label), '?')
    print(f"  Cluster {cluster_label} (tuned on {hero}) — {n_products} products")
    print(f"    WAPE={cm['WAPE']:.4f}, MASE={cm['MASE']:.4f}, MAE={cm['MAE']:.4f}, Bias={cm['Bias']:.4f}")



  PER-PRODUCT NORTH STAR METRICS
  Sorted by Bias (most under-predicted first)
              Product Cluster   WAPE   MASE     MAE    Bias    RMSE
        Festive Stack       2 1.0000 1.7556  2.7241 -2.7241  3.8011
        The Breakfast       1 0.1708 0.4666  5.8088 -1.9957  7.3526
               Bakery       1 0.1589 0.9332 10.0077 -1.6846 11.4323
         Tuna Toastie       3 0.6113 0.8151  1.5178 -1.0724  2.0509
     Avo, Egg & Bacon       0 0.5072 0.8966  1.7314 -0.9535  2.1592
        Black Pudding       0 0.6146 0.6891  1.5684 -0.8785  2.1606
      Buttd Mushrooms       0 0.6928 1.0392  1.5767 -0.8216  1.9345
              Egg Bap       0 0.8088 0.7077  1.3667 -0.7222  1.7996
            Fried Egg       0 0.4022 0.8345  3.4531 -0.7112  4.1461
          Baked Beans       0 0.3981 0.6694  2.1005 -0.6626  2.5655
                Bacon       0 0.3973 0.6481  2.3017 -0.5330  3.1436
          Poached Egg       0 0.5835 0.6781  1.7303 -0.5065  2.3532
     Veggie Breakfast       0 0.5870

In [9]:
# ==========================================
# 9. SAVE RESULTS TO CSV + SQLITE
# ==========================================
# Labelled 'Prophet_Hourly_Clustered' so it's distinguishable from the
# original 'Prophet_Hourly' and the daily models in the database.

import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_Prophet_Hourly_Clustered'

print(f"Saving results for run: {run_id}")

# --- CSV ---
product_leaderboard_df.to_csv('../results/prophet_hourly_per_product_results.csv', index=False)
daily_rollup[['Product_Name', 'Date_Only', 'Sales', 'Prophet_Prediction', 'Cluster']].to_csv(
    '../results/prophet_hourly_daily_rollup.csv', index=False)
print("  CSVs saved to ../results/")

# --- SQLite ---
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT,
            model_type TEXT,
            target_date TEXT,
            prediction_made_on TEXT,
            product_name TEXT,
            actual_sales REAL,
            predicted_sales REAL
        )
    """)

    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT,
            model_type TEXT,
            product_name TEXT,
            evaluation_horizon TEXT,
            WAPE REAL,
            MASE REAL,
            MAE REAL,
            Bias REAL
        )
    """)

    # Save every individual prediction
    predictions_for_db = daily_rollup[['Date_Only', 'Product_Name', 'Sales', 'Prophet_Prediction']].copy()
    predictions_for_db = predictions_for_db.rename(columns={
        'Date_Only': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'Prophet_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'Prophet_Hourly_Clustered'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    # Save aggregated metrics at each time horizon
    evaluation_start = pd.to_datetime('2025-11-02').date()
    evaluation_week_end = pd.to_datetime('2025-11-08').date()
    evaluation_month_end = pd.to_datetime('2025-11-30').date()
    horizon_products = all_products

    month_slice = daily_rollup[(daily_rollup['Date_Only'] >= evaluation_start) & (daily_rollup['Date_Only'] <= evaluation_month_end)].copy()
    week_slice = month_slice[month_slice['Date_Only'] <= evaluation_week_end].copy()
    day_slice = month_slice[month_slice['Date_Only'] == evaluation_start].copy()
    horizons = {'1-Day': day_slice, '1-Week': week_slice, '1-Month': month_slice}

    metrics_rows = []

    for product_name in horizon_products:
        for horizon_label, horizon_df in horizons.items():
            product_horizon_data = horizon_df[horizon_df['Product_Name'] == product_name] if len(horizon_df) > 0 else horizon_df.iloc[0:0]
            if len(product_horizon_data) > 0:
                m_dict = calculate_north_star_metrics(product_horizon_data)
                metrics_rows.append({
                    'run_id': run_id,
                    'model_type': 'Prophet_Hourly_Clustered',
                    'product_name': product_name,
                    'evaluation_horizon': horizon_label,
                    'WAPE': m_dict.get('WAPE', np.nan),
                    'MASE': m_dict.get('MASE', np.nan),
                    'MAE': m_dict.get('MAE', np.nan),
                    'Bias': m_dict.get('Bias', np.nan),
                })

    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) > 0:
            m_dict = calculate_north_star_metrics(horizon_df)
            metrics_rows.append({
                'run_id': run_id,
                'model_type': 'Prophet_Hourly_Clustered',
                'product_name': 'ALL_PRODUCTS',
                'evaluation_horizon': horizon_label,
                'WAPE': m_dict.get('WAPE', np.nan),
                'MASE': m_dict.get('MASE', np.nan),
                'MAE': m_dict.get('MAE', np.nan),
                'Bias': m_dict.get('Bias', np.nan),
            })

    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone! To compare models in the database:")
print("  SELECT model_type, WAPE, MASE, MAE, Bias FROM metrics_summary")
print("    WHERE product_name='ALL_PRODUCTS' AND evaluation_horizon='1-Month'")
print("    ORDER BY WAPE")


Saving results for run: 20260419_1515_Prophet_Hourly_Clustered
  CSVs saved to ../results/
  Logged 1920 predictions to predictions_log
  Logged 195 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260419_1515_Prophet_Hourly_Clustered

Done! To compare models in the database:
  SELECT model_type, WAPE, MASE, MAE, Bias FROM metrics_summary
    WHERE product_name='ALL_PRODUCTS' AND evaluation_horizon='1-Month'
    ORDER BY WAPE
